In [20]:
import sys
!{sys.executable} -m pip install numpy pandas scipy scikit-learn matplotlib


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import os
os.chdir(r'C:\Users\nkris\Downloads')
print(os.getcwd())

C:\Users\nkris\Downloads


In [22]:
# Cell 1 - Imports
import numpy as np
import pandas as pd
import pickle
import os
from scipy.stats import multivariate_normal


In [23]:
# Cell 2 - Load the dataset
df = pd.read_csv('dataset.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nLabel counts:")
print(df['label'].value_counts())

Shape: (1100, 5)

First 5 rows:
       width    spacing    x_coord    y_coord  label
0  41.968918  55.415902  -5.577101  10.044452    0.0
1  49.230640  45.360199   8.829402 -10.758954    0.0
2  57.787601  41.036929  -1.923631   8.367453    0.0
3  48.634041  50.051837 -13.665386   0.405630    0.0
4  48.305596  47.094536  10.231907  -2.229131    0.0

Label counts:
label
0.0    1000
1.0     100
Name: count, dtype: int64


In [24]:
# Cell 3 - Load Gaussian parameters from Member 1
with open('gaussian_params.pkl', 'rb') as f:
    gaussian_params = pickle.load(f)

print("Keys in gaussian_params:", list(gaussian_params.keys()))
print("\nFull contents:")
for key, value in gaussian_params.items():
    print(f"\n{key}:")
    print(value)

Keys in gaussian_params: ['safe', 'hotspot']

Full contents:

safe:
{'mu': array([4.98264012e+01, 4.98625513e+01, 4.21842398e-01, 3.25703004e-02]), 'sigma': array([[ 23.50757942,  -1.2013575 ,   0.12420324,  -0.53014293],
       [ -1.2013575 ,  26.93732452,   0.24680065,   0.7663428 ],
       [  0.12420324,   0.24680065, 104.83786786,   8.29489386],
       [ -0.53014293,   0.7663428 ,   8.29489386, 105.28018563]])}

hotspot:
{'mu': array([30.34232607, 20.06189587,  3.4498625 ,  1.51184713]), 'sigma': array([[ 19.73329459,   1.08084747,  -7.48093751, -13.09261654],
       [  1.08084747,  24.75242146,   0.25442665,   4.71597423],
       [ -7.48093751,   0.25442665, 112.11597123, -16.91687942],
       [-13.09261654,   4.71597423, -16.91687942,  95.79778764]])}


In [25]:
# Cell 4 - Extract parameters from Member 1's structure
mu_safe = gaussian_params['safe']['mu']
sigma_safe = gaussian_params['safe']['sigma']

mu_hotspot = gaussian_params['hotspot']['mu']
sigma_hotspot = gaussian_params['hotspot']['sigma']

print("mu_safe:", mu_safe)
print("mu_hotspot:", mu_hotspot)
print("\nsigma_safe shape:", sigma_safe.shape)
print("sigma_hotspot shape:", sigma_hotspot.shape)
print("\nAll parameters loaded successfully! ✅")

mu_safe: [4.98264012e+01 4.98625513e+01 4.21842398e-01 3.25703004e-02]
mu_hotspot: [30.34232607 20.06189587  3.4498625   1.51184713]

sigma_safe shape: (4, 4)
sigma_hotspot shape: (4, 4)

All parameters loaded successfully! ✅


In [26]:
# Cell 5 - Separate features and labels

X = df[['width', 'spacing', 'x_coord', 'y_coord']].values
y = df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nSample X[0]:", X[0])
print("Sample y[0]:", y[0])

X shape: (1100, 4)
y shape: (1100,)

Sample X[0]: [41.96891757 55.41590154 -5.57710131 10.04445157]
Sample y[0]: 0.0


In [27]:
# Cell 6 - Compute priors

num_samples = len(y)

num_hotspot = np.sum(y == 1)
num_safe = np.sum(y == 0)

P_hotspot = num_hotspot / num_samples
P_safe = num_safe / num_samples

print("P(hotspot):", P_hotspot)
print("P(safe):", P_safe)

P(hotspot): 0.09090909090909091
P(safe): 0.9090909090909091


In [28]:
# Cell 7 - Create Gaussian distributions

gaussian_safe = multivariate_normal(
    mean=mu_safe,
    cov=sigma_safe,
    allow_singular=True
)

gaussian_hotspot = multivariate_normal(
    mean=mu_hotspot,
    cov=sigma_hotspot,
    allow_singular=True
)

print("Gaussian models created successfully ✅")

Gaussian models created successfully ✅


In [29]:
# Cell 8 - Test likelihood on a single sample

x_sample = X[0]

likelihood_safe = gaussian_safe.logpdf(x_sample)
likelihood_hotspot = gaussian_hotspot.logpdf(x_sample)

print("x_sample:", x_sample)

print("\nlog P(x | safe):", likelihood_safe)
print("log P(x | hotspot):", likelihood_hotspot)

x_sample: [41.96891757 55.41590154 -5.57710131 10.04445157]

log P(x | safe): -14.024500434708797
log P(x | hotspot): -39.63270658795179


In [30]:
# Cell 9 - Convert priors to log space

log_P_hotspot = np.log(P_hotspot)
log_P_safe = np.log(P_safe)

print("log P(hotspot):", log_P_hotspot)
print("log P(safe):", log_P_safe)

log P(hotspot): -2.3978952727983707
log P(safe): -0.0953101798043249


In [31]:
# Cell 10 - Compute log posterior for one sample

# Already computed:
# likelihood_safe
# likelihood_hotspot

log_posterior_safe = likelihood_safe + log_P_safe
log_posterior_hotspot = likelihood_hotspot + log_P_hotspot

print("log posterior (safe):", log_posterior_safe)
print("log posterior (hotspot):", log_posterior_hotspot)

log posterior (safe): -14.119810614513122
log posterior (hotspot): -42.03060186075016


In [32]:
# Cell 11 - Convert log posterior to probability

# Stack values
log_vals = np.array([log_posterior_safe, log_posterior_hotspot])

# Stability trick
max_log = np.max(log_vals)
log_vals_shifted = log_vals - max_log

# Convert back to probabilities
probs = np.exp(log_vals_shifted)
probs = probs / np.sum(probs)

P_safe_given_x = probs[0]
P_hotspot_given_x = probs[1]

print("P(safe | x):", P_safe_given_x)
print("P(hotspot | x):", P_hotspot_given_x)

P(safe | x): 0.9999999999992439
P(hotspot | x): 7.559574932000147e-13


In [33]:
# Cell 12 - Batch inference + LLR-based GP targets

from sklearn.preprocessing import MinMaxScaler

# Batch log-likelihoods
log_likelihoods_safe    = gaussian_safe.logpdf(X)
log_likelihoods_hotspot = gaussian_hotspot.logpdf(X)

# Standard posterior (kept for reference)
log_posterior_safe_all    = log_likelihoods_safe    + log_P_safe
log_posterior_hotspot_all = log_likelihoods_hotspot + log_P_hotspot

log_vals_all = np.stack([log_posterior_safe_all,
                          log_posterior_hotspot_all], axis=1)
max_log_all      = log_vals_all.max(axis=1, keepdims=True)
log_vals_shifted = log_vals_all - max_log_all
probs_all        = np.exp(log_vals_shifted)
probs_all       /= probs_all.sum(axis=1, keepdims=True)
P_hotspot_all    = probs_all[:, 1]

# LLR-based targets (much smoother — better for GP)
LLR = log_likelihoods_hotspot - log_likelihoods_safe
LLR_scaled = MinMaxScaler().fit_transform(LLR.reshape(-1, 1)).ravel()

print("P_hotspot_all  — min:", P_hotspot_all.min().round(4),
                      "max:", P_hotspot_all.max().round(4),
                      "mean:", P_hotspot_all.mean().round(4))
print("LLR_scaled     — min:", LLR_scaled.min().round(4),
                      "max:", LLR_scaled.max().round(4),
                      "mean:", LLR_scaled.mean().round(4))

# Quick distribution check
import numpy as np
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
counts, _ = np.histogram(LLR_scaled, bins=bins)
print("\nLLR_scaled distribution:")
for i, c in enumerate(counts):
    print(f"  {bins[i]:.1f}–{bins[i+1]:.1f} : {'█' * (c // 10)} {c}")

P_hotspot_all  — min: 0.0 max: 1.0 mean: 0.091
LLR_scaled     — min: 0.0 max: 1.0 mean: 0.335

LLR_scaled distribution:
  0.0–0.1 :  6
  0.1–0.2 : ██████████ 105
  0.2–0.3 : █████████████████████████████████████████████ 450
  0.3–0.4 : █████████████████████████████████████ 379
  0.4–0.5 : █████ 58
  0.5–0.6 :  2
  0.6–0.7 :  7
  0.7–0.8 : ████ 43
  0.8–0.9 : ████ 44
  0.9–1.0 :  6


In [34]:
# Cell 13 - Fit Gaussian Process on (x_coord, y_coord) → LLR_scaled

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

X_spatial = X[:, 2:4]   # x_coord, y_coord only

# Tighter noise bound — forces GP to find spatial structure
kernel = (
    C(1.0, (1e-2, 1e2))
    * RBF(length_scale=[5.0, 5.0], length_scale_bounds=(0.5, 50.0))
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 0.5))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42
)

gp.fit(X_spatial, LLR_scaled)   # <-- LLR_scaled instead of P_hotspot_all

print("Optimised kernel:")
print(gp.kernel_)
print("\nLog-marginal-likelihood:", 
      gp.log_marginal_likelihood(gp.kernel_.theta).round(4))

Optimised kernel:
0.655**2 * RBF(length_scale=[0.5, 0.5]) + WhiteKernel(noise_level=0.5)

Log-marginal-likelihood: -1594.9987


c:\Users\nkris\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.5. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\nkris\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified lower bound 0.5. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\nkris\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.5. Increasing the bound and calling fit again may find a better value.
  warnings.war

In [35]:
# Cell 14 - Evaluate GP on a grid to build the risk surface

grid_res = 80   # 80×80 grid points

x_min, x_max = X_spatial[:, 0].min() - 5,  X_spatial[:, 0].max() + 5
y_min, y_max = X_spatial[:, 1].min() - 5,  X_spatial[:, 1].max() + 5

xx, yy      = np.meshgrid(np.linspace(x_min, x_max, grid_res),
                           np.linspace(y_min, y_max, grid_res))
X_grid      = np.column_stack([xx.ravel(), yy.ravel()])   # (6400, 2)

mu_grid, sigma_grid = gp.predict(X_grid, return_std=True)

# Clip to [0, 1] — GP can extrapolate slightly outside probability range
mu_grid    = np.clip(mu_grid,    0.0, 1.0).reshape(grid_res, grid_res)
sigma_grid = np.clip(sigma_grid, 0.0, 1.0).reshape(grid_res, grid_res)

print("Risk surface shape:", mu_grid.shape)
print("Predicted risk  — min:", mu_grid.min().round(4), "max:", mu_grid.max().round(4))
print("Uncertainty σ   — min:", sigma_grid.min().round(4), "max:", sigma_grid.max().round(4))

Risk surface shape: (80, 80)
Predicted risk  — min: 0.2209 max: 0.6047
Uncertainty σ   — min: 0.131 max: 0.1588


In [36]:
# Cell 15 - Plot risk surface (mean) + uncertainty + training points

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from pathlib import Path

# Robust project root detection (works regardless of Jupyter launch location)
current = Path().resolve()

for parent in [current] + list(current.parents):
    if (parent / "Hotspot_Detection").exists():
        project_root = parent / "Hotspot_Detection"
        break
else:
    raise Exception("Hotspot_Detection folder not found")

results_path = project_root / "results"
results_path.mkdir(exist_ok=True)

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

extent = [x_min, x_max, y_min, y_max]

# --- Panel 1: GP mean risk surface ---
ax1 = fig.add_subplot(gs[0])
im1 = ax1.imshow(mu_grid, origin='lower', extent=extent,
                  cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
ax1.scatter(X_spatial[y==0, 0], X_spatial[y==0, 1],
            c='white', edgecolors='gray', s=12, linewidths=0.5,
            alpha=0.5, label='safe')
ax1.scatter(X_spatial[y==1, 0], X_spatial[y==1, 1],
            c='red', edgecolors='black', s=30, linewidths=0.6,
            alpha=0.9, label='hotspot')
plt.colorbar(im1, ax=ax1, label='P(hotspot | x,y)')
ax1.set_title('GP mean risk surface', fontsize=11)
ax1.set_xlabel('x_coord'); ax1.set_ylabel('y_coord')
ax1.legend(fontsize=8, loc='upper right')

# --- Panel 2: GP uncertainty ---
ax2 = fig.add_subplot(gs[1])
im2 = ax2.imshow(sigma_grid, origin='lower', extent=extent,
                  cmap='Blues', aspect='auto')
plt.colorbar(im2, ax=ax2, label='Std dev σ(x,y)')
ax2.set_title('GP uncertainty (σ)', fontsize=11)
ax2.set_xlabel('x_coord'); ax2.set_ylabel('y_coord')

# --- Panel 3: Binary hotspot map at threshold 0.3 ---
threshold = 0.5
ax3 = fig.add_subplot(gs[2])
binary_map = (mu_grid >= threshold).astype(float)
ax3.imshow(binary_map, origin='lower', extent=extent,
           cmap='Reds', aspect='auto', alpha=0.6)
ax3.scatter(X_spatial[y==1, 0], X_spatial[y==1, 1],
            c='red', edgecolors='black', s=30, linewidths=0.6)
ax3.set_title(f'Predicted hotspot zones\n(threshold={threshold})', fontsize=11)
ax3.set_xlabel('x_coord'); ax3.set_ylabel('y_coord')

plt.suptitle('Gaussian Process Spatial Risk Surface — SPARC Hotspot Detection',
             fontsize=12, y=1.01)

save_path = results_path / "gp_risk_surface.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print("Saved to:", save_path)

plt.show()
print("Saved → gp_risk_surface.png")

Exception: Hotspot_Detection folder not found

In [18]:
# Cell 16 - Persist GP model for inference

import pickle, os
from pathlib import Path

# Same robust path logic
current = Path().resolve()

for parent in [current] + list(current.parents):
    if (parent / "Hotspot_Detection").exists():
        project_root = parent / "Hotspot_Detection"
        break
else:
    raise Exception("Hotspot_Detection folder not found")

results_path = project_root / "results"
results_path.mkdir(exist_ok=True)

with open(results_path / "gp_model.pkl", 'wb') as f:
    pickle.dump({
        'gp':         gp,
        'mu_grid':    mu_grid,
        'sigma_grid': sigma_grid,
        'grid_extent': [x_min, x_max, y_min, y_max],
        'grid_res':   grid_res,
    }, f)

print("GP model saved → gp_model.pkl ✅")

# Quick sanity check: query a single new layout point
x_new = np.array([[5.0, -8.0]])   # (x_coord, y_coord)
mu_new, sigma_new = gp.predict(x_new, return_std=True)
print(f"\nRisk at (5.0, -8.0)  →  μ={mu_new[0]:.4f},  σ={sigma_new[0]:.4f}")

GP model saved → ../results/gp_model.pkl ✅

Risk at (5.0, -8.0)  →  μ=0.3596,  σ=0.1480
